#### Model Development and Evaluation

#### Objective

The objective of this notebook is to train and evaluate machine learning models for predicting book popularity using features engineered during the preprocessing stage.

The dataset includes:
- Genre-based features obtained through multi-label encoding
- Text features extracted from book descriptions using TF-IDF
- Publication year
- Page count

The target variable is the Popularity_Score, derived from the logarithm of the rating count.

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor



In [8]:
# Load PreProcessed Dataset
df = pd.read_csv("preprocessed_books.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'preprocessed_books.csv'

In [ ]:
df.head()

,Page_Count,Popularity_Score,Pub_Year,fiction,fantasy,romance,young adult,historical fiction,nonfiction,mystery,...,wa,want,war,way,woman,work,world,year,york,young
0,2.088682,15.242884,-0.583660,1,1,0,1,0,0,0,...,0.120108,0.0,0.000000,0.141846,0.0,0.0,0.228578,0.0,0.0,0.0
1,0.037712,16.126869,-0.388194,1,1,0,1,0,0,0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
2,0.022705,15.081943,-1.365522,1,0,0,0,1,0,0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.244259,0.0,0.0,0.0
3,1.128228,14.889819,-0.974591,1,0,0,1,1,0,0,...,0.152579,0.0,0.000000,0.000000,0.0,0.0,0.145186,0.0,0.0,0.0
4,1.528417,15.127649,-0.974591,1,1,0,1,0,0,0,...,0.000000,0.0,0.261174,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0


#### Feature / Target Split

The target variable used for prediction is Popularity_Score. All remaining columns were treated as input features for model training.

In [ ]:
X = df.drop(columns=["Popularity_Score"])

y = df["Popularity_Score"]

In [ ]:
X.columns


Index(['Page_Count', 'Pub_Year', 'fiction', 'fantasy', 'romance',
       'young adult', 'historical fiction', 'nonfiction', 'mystery',
       'paranormal',
       ...
       'wa', 'want', 'war', 'way', 'woman', 'work', 'world', 'year', 'york',
       'young'],
      dtype='str', length=114)

#### Train-Test Split
The dataset was divided into training and testing sets to evaluate model performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Linear Regression Model

In [ ]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [ ]:
y_pred_lr = lr_model.predict(X_test)

In [ ]:
lr_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred_lr
})

lr_results

,Actual,Predicted
1760,10.440887,8.624090
3326,4.304065,4.748024
1770,12.842705,12.061286
3176,9.747360,11.844854
2099,10.234588,10.734631
...,...,...
2510,10.529880,8.223329
2752,10.698379,10.319695
1869,4.025352,1.931527
423,12.350658,10.954504


In [ ]:
# Evaluate Linear Regression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
lr_mae = mean_absolute_error(y_test, y_pred_lr)

lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)
print("R2 Score:", lr_r2)

Linear Regression Performance
MAE: 1.8117190480420218
RMSE: 2.2858539208277255
R2 Score: 0.5066378917375074


Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
rf_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred_rf
})

rf_results

,Actual,Predicted
1760,10.440887,7.522620
3326,4.304065,3.147272
1770,12.842705,12.462135
3176,9.747360,10.586927
2099,10.234588,10.695670
...,...,...
2510,10.529880,7.983594
2752,10.698379,9.315624
1869,4.025352,2.906875
423,12.350658,11.362622


In [ ]:
rf_mae = mean_absolute_error(y_test, y_pred_rf)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Performance")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2 Score:", rf_r2)

Random Forest Performance
MAE: 1.6746736735575465
RMSE: 2.19810794223221
R2 Score: 0.5437878198086097


 XGBoost Regressor

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

In [ ]:
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)

xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

xgb_r2 = r2_score(y_test, y_pred_xgb)

print("XGBoost Performance")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R2 Score:", xgb_r2)

XGBoost Performance
MAE: 1.6871787578914996
RMSE: 2.1800625400959004
R2 Score: 0.5512476351718528


In [ ]:
xgb_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_xgb
})

xgb_results.head(10)

,Actual,Predicted
0,10.440887,8.671866
1,4.304065,3.506866
2,12.842705,11.457658
3,9.747360,10.943945
4,10.234588,10.801654
5,12.152867,10.829464
6,4.189655,7.918324
7,13.150762,9.930892
8,12.294825,11.078169
9,5.641907,3.474185


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"],
    "MAE": [ lr_mae , rf_mae , xgb_mae],
    "RMSE": [ lr_rmse , rf_rmse, xgb_rmse],
    "R2 Score": [ lr_r2 , rf_r2 , xgb_r2 ]
})

comparison

,Model,MAE,RMSE,R2 Score
0,Linear Regression,1.811719,2.285854,0.506638
1,Random Forest,1.674674,2.198108,0.543788
2,XGBoost,1.687179,2.180063,0.551248


Three regression algorithms were evaluated for predicting book popularity.

Linear Regression served as a baseline model and achieved an R² score of 0.507. Ensemble tree-based methods produced stronger performance, indicating the presence of nonlinear relationships within the dataset.

Random Forest improved predictive accuracy, achieving an R² score of 0.544. XGBoost delivered the best overall performance with an R² score of 0.551 and the lowest RMSE value, making it the most effective model for predicting book popularity in this study.

Based on model evaluation results, XGBoost Regressor was selected as the final model for predicting book popularity.

In [ ]:
import joblib

joblib.dump(xgb_model, "xgboost_model.pkl")

['xgboost_model.pkl']

In [ ]:
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

['feature_columns.pkl']



To support future deployment, the final model and preprocessing components are saved. These files will be reused in the Flask application to ensure that new user inputs are transformed consistently before prediction.